In [12]:
!pip install rdkit
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [13]:
import joblib
import pandas as pd

# Load model 
model = joblib.load("/kaggle/input/datasets/krisnguyen123/vschobainho/model_3.pkl")

# Load screening features
screen_df = pd.read_parquet("/kaggle/input/datasets/krisnguyen123/vschobainho/gvae_screening_features.parquet")

# Load training feature columns
train_df = pd.read_parquet("/kaggle/input/datasets/krisnguyen123/vschobainho/wildtype egfr_features.parquet")
ecfp_cols = [c for c in train_df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in train_df.columns if c.startswith("Pharm_")]
train_features = ecfp_cols + pharm_cols

# Align feature space
X_screen = screen_df.reindex(columns=train_features, fill_value=0)

# Predict probability
proba = model.predict_proba(X_screen)[:, 1]

screen_df["Potent_probability"] = proba

# Keep only metadata + prediction
cols_to_keep = ["SMILES", "Potent_probability"]

if "BATCH" in screen_df.columns:
    cols_to_keep.append("BATCH")

if "InChIKey" in screen_df.columns:
    cols_to_keep.append("InChIKey")
    
top_hits = screen_df.sort_values("Potent_probability", ascending=False)
output_df = top_hits[cols_to_keep]

# Export
output_df.to_csv("virtual_screening_results.csv", index=False)

print("Saved: virtual_screening_results.csv")

Saved: virtual_screening_results.csv
